# 04 — Final combined RQ4: early-warning capability

**RQ4:** Can VLMs provide sufficiently early and actionable warnings before unsafe events occur?

This notebook combines the established HuRoN, PAL, and CrowdBot RQ4 procedures without changing their label policies. The authoritative `predicted_label` comes from the 96 combined clean files; probability argmax is never recomputed here. The corresponding 2,400 raw bag files are used only to recover the physical robot speed at each warning frame.

For each contiguous ground-truth `unsafe` event within a bag, the first preceding prediction of `potentially_unsafe` or `unsafe`, after the preceding unsafe event and strictly before the current event, is the warning:

\[
LeadTime = T_{unsafe} - T_{first\ warning}
\]

\[
T_{required} = T_{reaction} + \frac{v}{a_{brake}} + T_{margin},
\qquad T_{earliest}=T_{required}+A
\]

with \(T_{reaction}=0.2\) s, \(a_{brake}=1.0\) m/s², \(T_{margin}=0.5\) s, and \(A\in\{1,2,3,4,5\}\) s.

\[
Outcome =
\begin{cases}
Missed, & \text{no preceding warning} \\
TooLate, & LeadTime<T_{required} \\
Actionable, & T_{required}\le LeadTime\le T_{earliest} \\
TooEarly, & LeadTime>T_{earliest}
\end{cases}
\]

Runtime validation requires **49 HuRoN + 18 PAL + 19 CrowdBot = 86 unsafe events** in every one of the 96 model–approach–history configurations. Pooled event rates use all 86 events; case-study and equally weighted case-balanced summaries are also saved. Wilson intervals are descriptive event-level intervals and do not model within-bag dependence.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = next((
    p for p in [HERE, *HERE.parents]
    if (p / "case_studies").is_dir() and (p / "analysis").is_dir()
), None)

if ROOT is None:
    raise RuntimeError("Could not locate repository root.")

CLEAN = ROOT / "analysis" / "combined" / "clean"
OUT = ROOT / "results" / "rq4"
TEX = OUT / "latex"
OUT.mkdir(parents=True, exist_ok=True)
TEX.mkdir(parents=True, exist_ok=True)

MODELS = ["qwen", "internvl", "llava"]
APPROACHES = ["appfr", "appod"]
HISTORIES = list(range(2, 33, 2))
HISTORY_LABELS = [f"H{h:02d}" for h in HISTORIES]
CASE_STUDIES = ["huron", "pal", "crowdbot"]
INTERVALS = list(range(1, 6))
LABELS = ["safe", "potentially_unsafe", "unsafe"]
WARNING_LABELS = {"potentially_unsafe", "unsafe"}
OUTCOMES = ["missed", "too_late", "actionable", "too_early"]

EXPECTED_ROWS = 34_912
EXPECTED_BAGS = 25
EXPECTED_CASE_ROWS = {"huron": 16_862, "pal": 9_249, "crowdbot": 8_801}
EXPECTED_CASE_BAGS = {"huron": 15, "pal": 3, "crowdbot": 7}
EXPECTED_CASE_EVENTS = {"huron": 49, "pal": 18, "crowdbot": 19}
EXPECTED_EVENTS = sum(EXPECTED_CASE_EVENTS.values())
EXPECTED_CONFIGURATIONS = len(MODELS) * len(APPROACHES) * len(HISTORIES)
EXPECTED_METADATA_FILES = len(CASE_STUDIES)

T_REACTION, A_BRAKE, T_MARGIN = 0.2, 1.0, 0.5
MODEL_NAMES = {"qwen": "Qwen", "internvl": "InternVL", "llava": "LLaVA"}
APPROACH_NAMES = {"appfr": "AppFr", "appod": "AppOd"}
CASE_NAMES = {"huron": "HuRoN", "pal": "PAL", "crowdbot": "CrowdBot"}
EXPORTED = []

METADATA_FILES = {
    case_study: (
        ROOT
        / "case_studies"
        / case_study
        / "metadata"
        / "rq4_frame_metadata.csv"
    )
    for case_study in CASE_STUDIES
}

def clean_file(model, approach, history):
    candidates = [
        CLEAN / model / f"{model}_{approach}_h{history:02d}.csv",
        CLEAN / model / approach / f"H{history:02d}.csv",
        CLEAN / model / approach / f"h{history:02d}.csv",
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError("Missing combined clean file; tried: " + ", ".join(map(str, candidates)))

def normalize_label(series):
    return (series.astype("string").str.strip().str.lower()
            .str.replace("-", "_", regex=False).str.replace(" ", "_", regex=False)
            .str.replace(r"potentially_+unsafe", "potentially_unsafe", regex=True))

def normalize_id(series):
    return series.astype("string").str.strip().str.replace(r"(?<=\d)\.0$", "", regex=True)

def unsafe_events(labels):
    unsafe = labels.eq("unsafe").to_numpy()
    starts = np.flatnonzero(unsafe & ~np.r_[False, unsafe[:-1]])
    ends = np.flatnonzero(unsafe & ~np.r_[unsafe[1:], False])
    return list(zip(starts, ends))

def wilson(count, total, z=1.959963984540054):
    if total <= 0:
        return np.nan, np.nan
    p, denominator = count / total, 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return center - half, center + half

def export(df, name, caption, index=False):
    df.to_csv(OUT / f"{name}.csv", index=index)
    kwargs = dict(index=index, escape=True, na_rep="--", float_format=lambda x: f"{x:.6g}",
                  caption=caption, label=f"tab:{name.replace('_', '-')}",
                  multicolumn=True, multicolumn_format="c")
    kwargs["longtable" if len(df) > 80 else "position"] = True if len(df) > 80 else "tbp"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        latex = df.to_latex(**kwargs)
    (TEX / f"{name}.tex").write_text(latex, encoding="utf-8")
    if name not in EXPORTED:
        EXPORTED.append(name)

print("Analysis root:", ROOT)
print("Combined clean input:", CLEAN)
print("RQ4 output:", OUT)
print("RQ4 metadata:", {k: str(v) for k, v in METADATA_FILES.items()})

## 1. Index the 2,400 raw bag files

Raw files are indexed by case study, model, approach, and history. They provide timestamps, ground truth for cross-checking, and speed only; their saved prediction labels are not used.

In [ ]:
metadata_parts = []
metadata_index_rows = []

for case_study in CASE_STUDIES:
    path = METADATA_FILES[case_study]

    assert path.is_file(), f"Missing RQ4 metadata file: {path}"

    frame = pd.read_csv(path, low_memory=False)

    required = {
        "case_study",
        "bag_name",
        "global_bag_id",
        "frame_id",
        "frame_time_s",
        "ground_truth",
        "speed_mps",
    }

    assert required.issubset(frame), (
        f"{path}: missing {sorted(required - set(frame))}"
    )

    frame["case_study"] = normalize_label(frame["case_study"])
    frame["bag_name"] = frame["bag_name"].astype("string").str.strip()
    frame["global_bag_id"] = frame["global_bag_id"].astype("string").str.strip()
    frame["frame_id"] = normalize_id(frame["frame_id"])

    frame["frame_time_s"] = pd.to_numeric(
        frame["frame_time_s"], errors="coerce"
    )

    frame["speed_mps"] = pd.to_numeric(
        frame["speed_mps"], errors="coerce"
    )

    frame["ground_truth"] = normalize_label(frame["ground_truth"])

    expected_rows = EXPECTED_CASE_ROWS[case_study]
    expected_bags = EXPECTED_CASE_BAGS[case_study]

    assert len(frame) == expected_rows, (
        f"{case_study}: expected {expected_rows} metadata rows, "
        f"found {len(frame)}"
    )

    assert frame["global_bag_id"].nunique() == expected_bags
    assert not frame.duplicated(["global_bag_id", "frame_id"]).any()
    assert frame["frame_time_s"].notna().all()
    assert frame["speed_mps"].notna().all()
    assert set(frame["ground_truth"]) <= set(LABELS)

    metadata_parts.append(frame)

    metadata_index_rows.append({
        "case_study": case_study,
        "metadata_file": str(path.relative_to(ROOT)),
        "rows": len(frame),
        "bags": frame["global_bag_id"].nunique(),
    })

metadata = pd.concat(metadata_parts, ignore_index=True)

assert len(metadata) == EXPECTED_ROWS
assert metadata["global_bag_id"].nunique() == EXPECTED_BAGS
assert not metadata.duplicated(["global_bag_id", "frame_id"]).any()

metadata_index = pd.DataFrame(metadata_index_rows)

assert len(metadata_index) == EXPECTED_METADATA_FILES

metadata_index.to_csv(
    OUT / "rq4_metadata_index.csv",
    index=False,
)

export(
    metadata_index,
    "rq4_metadata_counts",
    "Compact frame metadata used for RQ4 timing and robot-speed calculations.",
)

display(metadata_index)

print(
    f"Loaded {len(metadata):,} frame-metadata rows "
    f"from {len(metadata_index)} case-study files."
)


## 2. Load, enrich, and validate all 96 combined configurations

Each combined clean configuration is joined one-to-one to its corresponding 25 raw bag files using case study, globally unique bag ID, and frame ID. Clean/raw time and ground truth must agree before an event is evaluated. Speed is interpolated only within its own bag, matching the separate RQ4 notebooks.

In [ ]:
def prepare_configuration(model, approach, history):
    path = clean_file(model, approach, history)
    clean = pd.read_csv(path, low_memory=False)

    required = {
        "case_study",
        "bag_name",
        "global_bag_id",
        "frame_id",
        "frame_time_s",
        "ground_truth",
        "predicted_label",
    }

    assert required.issubset(clean), (
        f"{path}: missing {sorted(required - set(clean))}"
    )

    clean["case_study"] = normalize_label(clean["case_study"])
    clean["bag_name"] = clean["bag_name"].astype("string").str.strip()
    clean["global_bag_id"] = clean["global_bag_id"].astype("string").str.strip()
    clean["frame_id"] = normalize_id(clean["frame_id"])

    clean["frame_time_s"] = pd.to_numeric(
        clean["frame_time_s"], errors="coerce"
    )

    clean["ground_truth"] = normalize_label(clean["ground_truth"])
    clean["predicted_label"] = normalize_label(clean["predicted_label"])

    name = f"{model} {approach} H{history:02d}"

    assert len(clean) == EXPECTED_ROWS
    assert clean["global_bag_id"].nunique() == EXPECTED_BAGS
    assert not clean.duplicated(["global_bag_id", "frame_id"]).any()
    assert clean["frame_time_s"].notna().all()
    assert set(clean["case_study"]) == set(CASE_STUDIES)
    assert set(clean["ground_truth"]) <= set(LABELS)
    assert set(clean["predicted_label"]) <= set(LABELS)

    assert (
        clean.groupby("case_study").size().astype(int).to_dict()
        == EXPECTED_CASE_ROWS
    )

    assert (
        clean.groupby("case_study")["global_bag_id"]
        .nunique()
        .astype(int)
        .to_dict()
        == EXPECTED_CASE_BAGS
    )

    meta = metadata[
        [
            "case_study",
            "global_bag_id",
            "frame_id",
            "frame_time_s",
            "ground_truth",
            "speed_mps",
            "source_file",
        ]
    ].copy()

    meta = meta.rename(columns={
        "frame_time_s": "raw_frame_time_s",
        "ground_truth": "raw_ground_truth",
        "source_file": "raw_source_file",
    })

    prepared = clean.merge(
        meta,
        on=["case_study", "global_bag_id", "frame_id"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    unmatched = int(prepared["_merge"].ne("both").sum())

    if unmatched:
        failed = prepared.loc[
            prepared["_merge"].ne("both"),
            ["case_study", "global_bag_id", "frame_id"],
        ]

        raise AssertionError(
            f"{name}: {unmatched} clean rows did not match RQ4 metadata; "
            f"examples={failed.head(5).to_dict('records')}"
        )

    assert len(prepared) == EXPECTED_ROWS

    max_time_difference = float(
        (
            prepared["frame_time_s"]
            - prepared["raw_frame_time_s"]
        )
        .abs()
        .max()
    )

    gt_disagreements = int(
        prepared["ground_truth"]
        .ne(prepared["raw_ground_truth"])
        .sum()
    )

    assert max_time_difference <= 1e-6, (
        f"{name}: time mismatch {max_time_difference}"
    )

    assert gt_disagreements == 0, (
        f"{name}: {gt_disagreements} GT disagreements"
    )

    assert prepared["speed_mps"].notna().all()

    prepared = prepared.sort_values(
        ["global_bag_id", "frame_time_s", "frame_id"]
    ).reset_index(drop=True)

    return prepared.drop(columns="_merge"), {
        "model": model,
        "approach": approach,
        "history": f"H{history:02d}",
        "history_frames": history,
        "clean_file": str(path.relative_to(ROOT)),
        "clean_rows": len(clean),
        "metadata_rows": len(metadata),
        "metadata_files": len(METADATA_FILES),
        "bags": clean["global_bag_id"].nunique(),
        "unmatched_rows": unmatched,
        "maximum_time_difference_s": max_time_difference,
        "ground_truth_disagreements": gt_disagreements,
        "missing_speed_values": int(prepared["speed_mps"].isna().sum()),
    }


print("Configuration preparation function is ready.")


## 3. Extract the first warning before each unsafe event

Unsafe events are detected independently inside every bag. Search windows never cross a bag boundary or a preceding unsafe event.

In [ ]:
event_rows, validation_rows = [], []
reference_identity = reference_bags = None

for config_number, (model, approach, history) in enumerate(
    ((m, a, h) for m in MODELS for a in APPROACHES for h in HISTORIES), start=1
):
    prepared, audit = prepare_configuration(model, approach, history)
    identity = (prepared[["case_study", "global_bag_id", "frame_id", "frame_time_s", "ground_truth"]]
                .sort_values(["global_bag_id", "frame_id"]).reset_index(drop=True))
    if reference_identity is None:
        reference_identity = identity.copy()
        reference_bags = prepared[["case_study", "bag_name", "global_bag_id"]].drop_duplicates().copy()
    else:
        assert identity[["case_study", "global_bag_id", "frame_id", "ground_truth"]].equals(
            reference_identity[["case_study", "global_bag_id", "frame_id", "ground_truth"]]
        ), f"{model} {approach} H{history:02d}: physical-frame identity changed"
        assert np.allclose(identity["frame_time_s"], reference_identity["frame_time_s"], atol=1e-6, rtol=0)

    case_event_counts = {}
    for global_bag_id, bag in prepared.groupby("global_bag_id", sort=True):
        bag = bag.sort_values(["frame_time_s", "frame_id"]).reset_index(drop=True)
        case_study, bag_name = str(bag.loc[0, "case_study"]), str(bag.loc[0, "bag_name"])
        previous_end_s = -np.inf
        bag_events = unsafe_events(bag["ground_truth"])
        case_event_counts[case_study] = case_event_counts.get(case_study, 0) + len(bag_events)

        for event_id, (start, end) in enumerate(bag_events):
            unsafe_start_s = float(bag.loc[start, "frame_time_s"])
            unsafe_end_s = float(bag.loc[end, "frame_time_s"])
            unsafe_start_frame_id = str(bag.loc[start, "frame_id"])
            unsafe_end_frame_id = str(bag.loc[end, "frame_id"])
            before = bag[(bag["frame_time_s"] > previous_end_s) & (bag["frame_time_s"] < unsafe_start_s)]
            warnings_before = before[before["predicted_label"].isin(WARNING_LABELS)]
            warning = warnings_before.iloc[0] if len(warnings_before) else None

            if warning is None:
                warning_frame_id = warning_label = warning_source_file = None
                warning_time = speed = lead_time = t_required = np.nan
            else:
                warning_frame_id = str(warning["frame_id"])
                warning_label = str(warning["predicted_label"])
                warning_source_file = str(warning["raw_source_file"])
                warning_time = float(warning["frame_time_s"])
                speed = float(warning["speed_mps"])
                lead_time = unsafe_start_s - warning_time
                t_required = T_REACTION + speed / A_BRAKE + T_MARGIN

            event_rows.append({
                "model": model, "approach": approach, "history": f"H{history:02d}",
                "history_frames": history, "case_study": case_study,
                "bag_name": bag_name, "global_bag_id": str(global_bag_id),
                "event_id_within_bag": event_id,
                "event_key": f"{global_bag_id}::{unsafe_start_frame_id}",
                "unsafe_start_frame_id": unsafe_start_frame_id,
                "unsafe_end_frame_id": unsafe_end_frame_id,
                "unsafe_start_s": unsafe_start_s, "unsafe_end_s": unsafe_end_s,
                "previous_unsafe_end_s": previous_end_s if np.isfinite(previous_end_s) else np.nan,
                "first_warning_frame_id": warning_frame_id, "first_warning_s": warning_time,
                "first_warning_label": warning_label, "speed_at_warning_mps": speed,
                "lead_time_s": lead_time, "t_required_s": t_required,
                "warning_raw_source_file": warning_source_file,
            })
            previous_end_s = unsafe_end_s

    audit.update({f"{case}_unsafe_events": case_event_counts.get(case, 0) for case in CASE_STUDIES})
    validation_rows.append(audit)
    if config_number % 16 == 0:
        print(f"Prepared {config_number:>2}/{EXPECTED_CONFIGURATIONS} configurations ({model})")

events = pd.DataFrame(event_rows)
validation = pd.DataFrame(validation_rows)
assert len(validation) == EXPECTED_CONFIGURATIONS
assert len(events) == EXPECTED_CONFIGURATIONS * EXPECTED_EVENTS
for case_study, expected in EXPECTED_CASE_EVENTS.items():
    assert validation[f"{case_study}_unsafe_events"].eq(expected).all()
assert events["event_key"].nunique() == EXPECTED_EVENTS
assert events["event_key"].value_counts().eq(EXPECTED_CONFIGURATIONS).all()
assert events.loc[events["first_warning_s"].notna(), "lead_time_s"].gt(0).all()

event_inventory = (events.query("model == 'qwen' and approach == 'appfr' and history == 'H02'")
                   [["case_study", "bag_name", "global_bag_id", "event_id_within_bag", "event_key",
                     "unsafe_start_frame_id", "unsafe_end_frame_id", "unsafe_start_s", "unsafe_end_s"]]
                   .sort_values(["case_study", "global_bag_id", "unsafe_start_s"]).reset_index(drop=True))
events_by_bag = (reference_bags.merge(
    event_inventory.groupby("global_bag_id").size().rename("unsafe_events"),
    left_on="global_bag_id", right_index=True, how="left")
    .assign(unsafe_events=lambda x: x["unsafe_events"].fillna(0).astype(int))
    .sort_values(["case_study", "global_bag_id"]).reset_index(drop=True))
event_distribution = (event_inventory.groupby("case_study")
                      .agg(bags_with_events=("global_bag_id", "nunique"), unsafe_events=("event_key", "size"))
                      .reindex(CASE_STUDIES).reset_index())
event_distribution["all_bags"] = event_distribution["case_study"].map(EXPECTED_CASE_BAGS)
event_distribution.loc[len(event_distribution)] = ["combined", events_by_bag.query("unsafe_events > 0")["global_bag_id"].nunique(),
                                                    EXPECTED_EVENTS, EXPECTED_BAGS]

export(validation, "rq4_configuration_validation",
       "Validation of the 96 combined RQ4 configurations.")
export(event_inventory, "rq4_unsafe_event_inventory",
       "Canonical combined unsafe-event inventory.")
export(events_by_bag, "rq4_unsafe_events_by_bag",
       "Unsafe-event counts for all 25 bags.")
export(event_distribution, "rq4_event_distribution",
       "Unsafe-event distribution across the three case studies.")
events.to_csv(OUT / "rq4_event_results.csv", index=False)

display(event_distribution)
print(f"Validated {len(events):,} event rows = {EXPECTED_CONFIGURATIONS} configurations × {EXPECTED_EVENTS} events.")

## 4. Classify warning timing for A = 1–5 seconds

Every configuration-event pair receives exactly one mutually exclusive outcome at each value of \(A\). Missed and too-late counts must remain fixed across \(A\); actionable counts can only increase and too-early counts can only decrease.

In [ ]:
all_intervals = events.merge(pd.DataFrame({"allowed_interval_s": INTERVALS}), how="cross")
all_intervals["t_earliest_s"] = all_intervals["t_required_s"] + all_intervals["allowed_interval_s"]
all_intervals["outcome"] = np.select(
    [all_intervals["lead_time_s"].isna(),
     all_intervals["lead_time_s"] < all_intervals["t_required_s"],
     all_intervals["lead_time_s"] <= all_intervals["t_earliest_s"]],
    ["missed", "too_late", "actionable"], default="too_early"
)

CONFIG_KEYS = ["model", "approach", "history", "history_frames", "allowed_interval_s"]

def outcome_summary(frame, keys):
    result = frame.groupby(keys + ["outcome"]).size().unstack(fill_value=0).reset_index()
    for outcome in OUTCOMES:
        if outcome not in result:
            result[outcome] = 0
    result["total_events"] = result[OUTCOMES].sum(axis=1)
    result["warning"] = result["total_events"] - result["missed"]
    for outcome in OUTCOMES + ["warning"]:
        result[f"{outcome}_rate"] = result[outcome] / result["total_events"]
        ci = result.apply(lambda row: wilson(int(row[outcome]), int(row["total_events"])), axis=1)
        result[f"{outcome}_lower"] = [value[0] for value in ci]
        result[f"{outcome}_upper"] = [value[1] for value in ci]
        result[f"{outcome}_result"] = result.apply(
            lambda row: (f'{int(row[outcome])}/{int(row["total_events"])} '
                         f'({100 * row[f"{outcome}_rate"]:.2f}%; 95% CI '
                         f'{100 * row[f"{outcome}_lower"]:.2f}–{100 * row[f"{outcome}_upper"]:.2f}%)'), axis=1)
    return result.sort_values(keys).reset_index(drop=True)

pooled_summary = outcome_summary(all_intervals, CONFIG_KEYS)
case_summary = outcome_summary(all_intervals, ["case_study", *CONFIG_KEYS])
rate_columns = [f"{outcome}_rate" for outcome in OUTCOMES + ["warning"]]
case_balanced_summary = (case_summary.groupby(CONFIG_KEYS)[rate_columns].mean().reset_index()
                         .rename(columns={column: f"case_balanced_{column}" for column in rate_columns}))

assert len(all_intervals) == EXPECTED_CONFIGURATIONS * EXPECTED_EVENTS * len(INTERVALS)
assert len(pooled_summary) == EXPECTED_CONFIGURATIONS * len(INTERVALS)
assert len(case_summary) == len(CASE_STUDIES) * EXPECTED_CONFIGURATIONS * len(INTERVALS)
assert pooled_summary["total_events"].eq(EXPECTED_EVENTS).all()
for case_study, expected in EXPECTED_CASE_EVENTS.items():
    assert case_summary.query("case_study == @case_study")["total_events"].eq(expected).all()
assert pooled_summary[OUTCOMES].sum(axis=1).eq(EXPECTED_EVENTS).all()

fixed = pooled_summary.groupby(["model", "approach", "history"]).agg(
    missed_values=("missed", "nunique"), late_values=("too_late", "nunique"),
    warning_values=("warning", "nunique"))
assert fixed.eq(1).all().all()
for _, part in pooled_summary.groupby(["model", "approach", "history"]):
    part = part.sort_values("allowed_interval_s")
    assert part["actionable"].diff().dropna().ge(0).all()
    assert part["too_early"].diff().dropna().le(0).all()
    assert (part["actionable"] + part["too_early"]).nunique() == 1
assert all_intervals.query("outcome == 'missed'")["lead_time_s"].isna().all()
assert (all_intervals.query("outcome == 'too_late'")["lead_time_s"] <
        all_intervals.query("outcome == 'too_late'")["t_required_s"]).all()
assert (all_intervals.query("outcome == 'actionable'")["lead_time_s"] >=
        all_intervals.query("outcome == 'actionable'")["t_required_s"]).all()
assert (all_intervals.query("outcome == 'actionable'")["lead_time_s"] <=
        all_intervals.query("outcome == 'actionable'")["t_earliest_s"]).all()
assert (all_intervals.query("outcome == 'too_early'")["lead_time_s"] >
        all_intervals.query("outcome == 'too_early'")["t_earliest_s"]).all()

all_intervals.to_csv(OUT / "rq4_all_intervals.csv", index=False)
export(pooled_summary, "rq4_pooled_outcome_summary",
       "Combined RQ4 outcome counts, rates, and Wilson intervals over 86 unsafe events.")
export(case_summary, "rq4_case_study_outcome_summary",
       "RQ4 outcome results separately within HuRoN, PAL, and CrowdBot.")
export(case_balanced_summary, "rq4_case_balanced_summary",
       "RQ4 outcome rates averaged equally across the three case studies.")
display(pooled_summary.head())
print(f"Validated {len(all_intervals):,} interval-event rows and {len(pooled_summary)} pooled summaries.")

## 5. Combined descriptive answers

`interval_independent` reports warning coverage, missed events, and too-late warnings once because they do not depend on \(A\). `interval_dependent` reports actionable and too-early outcomes at every \(A\). Best histories retain all ties.

In [ ]:
interval_independent = pooled_summary.query("allowed_interval_s == 1")[[
    "model", "approach", "history", "history_frames", "warning_result", "missed_result", "too_late_result"
]].reset_index(drop=True)
interval_dependent = pooled_summary[[
    "model", "approach", "history", "history_frames", "allowed_interval_s",
    "actionable_result", "too_early_result"
]].copy()

best_count = pooled_summary.groupby(["model", "approach", "allowed_interval_s"])["actionable"].transform("max")
best_histories = pooled_summary[pooled_summary["actionable"].eq(best_count)][[
    "model", "approach", "allowed_interval_s", "history", "history_frames",
    "actionable", "actionable_rate", "actionable_result", "missed", "too_late", "too_early"
]].sort_values(["model", "approach", "allowed_interval_s", "history_frames"]).reset_index(drop=True)

overall_rows = []
for allowed_interval in INTERVALS:
    part = pooled_summary.query("allowed_interval_s == @allowed_interval")
    maximum = int(part["actionable"].max())
    winners = part[part["actionable"].eq(maximum)]
    overall_rows.append({
        "allowed_interval_s": allowed_interval, "maximum_actionable": maximum,
        "total_events": EXPECTED_EVENTS, "maximum_actionable_rate": maximum / EXPECTED_EVENTS,
        "highest_result": f"{maximum}/{EXPECTED_EVENTS} ({100 * maximum / EXPECTED_EVENTS:.2f}%)",
        "winning_configurations": "; ".join(
            f"{MODEL_NAMES[row.model]} {APPROACH_NAMES[row.approach]} {row.history}"
            for row in winners.itertuples(index=False)), "number_of_ties": len(winners),
    })
overall_effect = pd.DataFrame(overall_rows)

profile_numeric_rows, profile_display_rows = [], []
for model in MODELS:
    for approach in APPROACHES:
        part = pooled_summary.query("model == @model and approach == @approach")
        numeric = {"model": model, "approach": approach,
                   "model_approach": f"{MODEL_NAMES[model]} {APPROACH_NAMES[approach]}"}
        shown = {"Model–approach": numeric["model_approach"]}
        for allowed_interval in INTERVALS:
            interval = part.query("allowed_interval_s == @allowed_interval")
            for outcome in ["actionable", "too_early"]:
                numeric[f"A{allowed_interval}_{outcome}_mean_count"] = interval[outcome].mean()
                numeric[f"A{allowed_interval}_{outcome}_mean_rate"] = interval[f"{outcome}_rate"].mean()
            shown[f"A = {allowed_interval} s"] = (
                f'{100 * numeric[f"A{allowed_interval}_actionable_mean_rate"]:.1f}% / '
                f'{100 * numeric[f"A{allowed_interval}_too_early_mean_rate"]:.1f}%')
        representative = part.query("allowed_interval_s == 1")
        for outcome in ["too_late", "missed", "warning"]:
            numeric[f"{outcome}_mean_count"] = representative[outcome].mean()
            numeric[f"{outcome}_mean_rate"] = representative[f"{outcome}_rate"].mean()
        shown["Too late"] = f'{100 * numeric["too_late_mean_rate"]:.1f}%'
        shown["Missed"] = f'{100 * numeric["missed_mean_rate"]:.1f}%'
        shown["Warning coverage"] = f'{100 * numeric["warning_mean_rate"]:.1f}%'
        profile_numeric_rows.append(numeric)
        profile_display_rows.append(shown)

warning_profile_numeric = pd.DataFrame(profile_numeric_rows)
warning_profile = pd.DataFrame(profile_display_rows).set_index("Model–approach")
case_warning_profile = (case_summary.groupby(["case_study", "model", "approach", "allowed_interval_s"])
                        [["actionable_rate", "too_early_rate", "too_late_rate", "missed_rate", "warning_rate"]]
                        .mean().reset_index())

export(interval_independent, "rq4_interval_independent",
       "Combined warning coverage, missed, and too-late results.")
export(interval_dependent, "rq4_interval_dependent",
       "Combined actionable and too-early results at A equals 1 through 5 seconds.")
export(best_histories, "rq4_best_histories",
       "Best combined actionable histories at each model, approach, and A; all ties retained.")
export(overall_effect, "rq4_overall_effect_of_a",
       "Highest combined actionable result observed at each value of A.")
export(warning_profile_numeric, "rq4_complete_warning_profile_numeric",
       "Mean combined outcome counts and rates across 16 histories.")
export(warning_profile, "rq4_complete_warning_profile",
       "Mean combined rates across 16 histories; A cells show actionable / too early.", index=True)
export(case_warning_profile, "rq4_case_study_warning_profile",
       "Mean RQ4 rates across histories within each case study.")

display(overall_effect)
display(warning_profile)
display(best_histories)

## 6. Publication tables

The three actionable tables report `count/86 (rate)` for every history, approach, and \(A\). Separate combined tables report too-early, missed, and too-late outcomes. The case-study table retains the correct denominator for each dataset (49, 18, or 19).

In [ ]:
def count_percent(row, outcome):
    return f'{int(row[outcome])}/{int(row["total_events"])} ({100 * row[f"{outcome}_rate"]:.2f}%)'

actionable_tables, too_early_tables = {}, {}
column_order = pd.MultiIndex.from_product([INTERVALS, ["AppFr", "AppOd"]])
for model in MODELS:
    part = pooled_summary.query("model == @model").copy()
    part["Approach"] = part["approach"].map(APPROACH_NAMES)
    for outcome, destination in [("actionable", actionable_tables), ("too_early", too_early_tables)]:
        part["Result"] = part.apply(lambda row: count_percent(row, outcome), axis=1)
        table = part.pivot(index="history", columns=["allowed_interval_s", "Approach"], values="Result")
        table = table.reindex(index=HISTORY_LABELS, columns=column_order)
        table.columns = pd.MultiIndex.from_tuples(
            [(f"A = {allowed_interval} s", approach) for allowed_interval, approach in table.columns],
            names=["Allowed interval", "Approach"])
        table.index.name = "History"
        destination[model] = table
        export(table, f"rq4_{model}_{outcome}_paper_table",
               f"Combined {outcome.replace('_', ' ')} warnings for {MODEL_NAMES[model]}.", index=True)

fixed_data = pooled_summary.query("allowed_interval_s == 1").copy()
fixed_data["Model"] = fixed_data["model"].map(MODEL_NAMES)
fixed_data["Approach"] = fixed_data["approach"].map(APPROACH_NAMES)
fixed_tables = {}
for outcome in ["missed", "too_late"]:
    fixed_data["Result"] = fixed_data.apply(lambda row: count_percent(row, outcome), axis=1)
    table = fixed_data.pivot(index="history", columns=["Model", "Approach"], values="Result")
    ordered = pd.MultiIndex.from_product([[MODEL_NAMES[m] for m in MODELS], ["AppFr", "AppOd"]])
    table = table.reindex(index=HISTORY_LABELS, columns=ordered)
    table.index.name = "History"
    fixed_tables[outcome] = table
    export(table, f"rq4_{outcome}_paper_table",
           f"Combined {outcome.replace('_', ' ')} outcomes; independent of A.", index=True)

case_paper = case_summary.copy()
case_paper["Result"] = case_paper.apply(lambda row: count_percent(row, "actionable"), axis=1)
case_paper = (case_paper.pivot(index=["case_study", "model", "approach", "history"],
                              columns="allowed_interval_s", values="Result")
              .reindex(columns=INTERVALS).reset_index())
case_paper.columns = [*case_paper.columns[:4], *[f"A = {a} s" for a in INTERVALS]]
export(case_paper, "rq4_actionable_by_case_study_paper_table",
       "Actionable warning results within each case study, model, approach, and history.")

for model in MODELS:
    print(f"\n{MODEL_NAMES[model]} — combined actionable warnings")
    display(actionable_tables[model])
display(fixed_tables["missed"])
display(fixed_tables["too_late"])

## 7. Final export manifest and checks

In [ ]:
CSV_ONLY = ["rq4_metadata_index", "rq4_event_results", "rq4_all_intervals"]
paired_names = sorted(set(EXPORTED + ["rq4_output_manifest"]))
manifest = pd.DataFrame(
    [{"table": name, "csv": str(OUT / f"{name}.csv"), "latex": str(TEX / f"{name}.tex"),
      "format": "CSV + LaTeX"} for name in paired_names]
    + [{"table": name, "csv": str(OUT / f"{name}.csv"),
        "latex": "not generated for row-level/index extract", "format": "CSV only"}
       for name in CSV_ONLY]
)
export(manifest, "rq4_output_manifest", "Manifest of combined RQ4 result files.")

missing_csv = [name for name in set(EXPORTED + CSV_ONLY) if not (OUT / f"{name}.csv").is_file()]
missing_tex = [name for name in set(EXPORTED) if not (TEX / f"{name}.tex").is_file()]
assert not missing_csv and not missing_tex, (missing_csv, missing_tex)
assert len(validation) == 96
assert len(metadata_index) == 3
assert len(event_inventory) == 86
assert len(events) == 8_256
assert len(all_intervals) == 41_280
assert len(pooled_summary) == 480
assert len(case_summary) == 1_440

print("COMBINED RQ4 COMPLETE")
print("Validated clean configurations:", len(validation))
print("RQ4 metadata files used for timing/speed:", len(metadata_index))
print("Unsafe events per configuration:", len(event_inventory), "= 49 HuRoN + 18 PAL + 19 CrowdBot")
print("Event-result rows:", len(events))
print("Interval-event rows:", len(all_intervals))
print("Output folder:", OUT)
display(manifest)

In [ ]:
# ============================================================
# Cross-RQ analysis:
# RQ4 warning outcomes for histories significant in BOTH
# RQ3 Accuracy and RQ3 Macro-F1
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. Locate Analysis folder
# ------------------------------------------------------------
HERE = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [HERE, *HERE.parents]
        if (p / "case_studies").is_dir()
        and (p / "analysis").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate repository root.")

RQ3 = ROOT / "results" / "rq3"
RQ4 = ROOT / "results" / "rq4"
TEX = RQ4 / "latex"
TEX.mkdir(parents=True, exist_ok=True)

KEYS = ["model", "approach", "history"]

MODEL_NAMES = {
    "qwen": "Qwen",
    "internvl": "InternVL",
    "llava": "LLaVA",
}

APPROACH_NAMES = {
    "appfr": "AppFr",
    "appod": "AppOd",
}

# ------------------------------------------------------------
# 2. Read RQ3 significant-history wins
# ------------------------------------------------------------
accuracy_wins = pd.read_csv(
    RQ3 / "rq3_accuracy_significant_history_wins.csv"
)

macro_f1_wins = pd.read_csv(
    RQ3 / "rq3_macro_f1_significant_history_wins.csv"
)

accuracy_wins = accuracy_wins[
    KEYS + ["significant_wins", "significant_losses", "win_tier"]
].rename(columns={
    "significant_wins": "accuracy_significant_wins",
    "significant_losses": "accuracy_significant_losses",
    "win_tier": "accuracy_win_tier",
})

macro_f1_wins = macro_f1_wins[
    KEYS + ["significant_wins", "significant_losses", "win_tier"]
].rename(columns={
    "significant_wins": "macro_f1_significant_wins",
    "significant_losses": "macro_f1_significant_losses",
    "win_tier": "macro_f1_win_tier",
})

# Inner join = significant winner for both Accuracy and Macro-F1
common_histories = accuracy_wins.merge(
    macro_f1_wins,
    on=KEYS,
    how="inner",
    validate="one_to_one",
)

common_histories["history_frames"] = (
    common_histories["history"].str.extract(r"(\d+)")[0].astype(int)
)

common_histories["model_order"] = common_histories["model"].map(
    {"qwen": 0, "internvl": 1, "llava": 2}
)
common_histories["approach_order"] = common_histories["approach"].map(
    {"appfr": 0, "appod": 1}
)

common_histories = (
    common_histories
    .sort_values(["model_order", "approach_order", "history_frames"])
    .drop(columns=["model_order", "approach_order"])
    .reset_index(drop=True)
)

if common_histories.empty:
    raise ValueError(
        "No histories had a significant Wilcoxon win for both metrics."
    )

# ------------------------------------------------------------
# 3. Join common histories with pooled combined RQ4 outcomes
# ------------------------------------------------------------
rq4_outcomes = pd.read_csv(
    RQ4 / "rq4_pooled_outcome_summary.csv"
)

required = {
    *KEYS,
    "allowed_interval_s",
    "total_events",
    "actionable",
    "actionable_rate",
    "too_early",
    "too_early_rate",
    "too_late",
    "too_late_rate",
    "missed",
    "missed_rate",
    "warning",
    "warning_rate",
}

missing = required.difference(rq4_outcomes.columns)
if missing:
    raise ValueError(f"Missing RQ4 columns: {sorted(missing)}")

selected = common_histories.merge(
    rq4_outcomes,
    on=KEYS,
    how="left",
    validate="one_to_many",
)

selected["allowed_interval_s"] = selected["allowed_interval_s"].astype(int)

# Each selected history should have A = 1, 2, 3, 4, 5 seconds
counts_per_history = selected.groupby(KEYS).size()
assert counts_per_history.eq(5).all(), counts_per_history
assert set(selected["allowed_interval_s"]) == {1, 2, 3, 4, 5}

long_columns = [
    "model", "approach", "history",
    "accuracy_significant_wins",
    "macro_f1_significant_wins",
    "accuracy_win_tier",
    "macro_f1_win_tier",
    "allowed_interval_s",
    "total_events",
    "actionable", "actionable_rate",
    "too_early", "too_early_rate",
    "too_late", "too_late_rate",
    "missed", "missed_rate",
    "warning", "warning_rate",
]

selected_long = selected[long_columns].copy()

# ------------------------------------------------------------
# 4. Compact publication table
# ------------------------------------------------------------
def count_rate(row, count_column):
    total = int(row["total_events"])
    count = int(row[count_column])
    percentage = 100 * count / total if total else np.nan
    return f"{count}/{total} ({percentage:.2f}%)"


publication_rows = []

for (model, approach, history), group in selected_long.groupby(
    KEYS, sort=False
):
    by_interval = group.set_index("allowed_interval_s")
    fixed = by_interval.loc[1]

    row = {
        "Model": MODEL_NAMES.get(model, model),
        "Approach": APPROACH_NAMES.get(approach, approach),
        "History": history,
        "Accuracy significant wins":
            int(fixed["accuracy_significant_wins"]),
        "Macro-F1 significant wins":
            int(fixed["macro_f1_significant_wins"]),
    }

    # Actionable and too-early depend on the allowed interval A
    for interval in range(1, 6):
        result = by_interval.loc[interval]
        row[f"A={interval}s: Actionable / Too early"] = (
            f"{count_rate(result, 'actionable')} / "
            f"{count_rate(result, 'too_early')}"
        )

    # These outcomes do not change with A
    row["Too late"] = count_rate(fixed, "too_late")
    row["Missed"] = count_rate(fixed, "missed")
    row["Warning coverage"] = count_rate(fixed, "warning")

    publication_rows.append(row)

publication_table = pd.DataFrame(publication_rows)

# ------------------------------------------------------------
# 5. Export CSV and LaTeX
# ------------------------------------------------------------
def export_both(table, name, caption):
    csv_path = RQ4 / f"{name}.csv"
    tex_path = TEX / f"{name}.tex"

    table.to_csv(csv_path, index=False)
    table.to_latex(
        tex_path,
        index=False,
        escape=True,
        caption=caption,
        label=f"tab:{name.replace('_', '-')}",
        na_rep="--",
        float_format=lambda value: f"{value:.4f}",
        longtable=len(table) > 60,
    )

    return csv_path, tex_path


export_both(
    common_histories,
    "rq3_common_significant_histories",
    "Histories with at least one Holm-significant Wilcoxon win "
    "for both accuracy and macro-F1.",
)

export_both(
    selected_long,
    "rq3_common_histories_rq4_outcomes_long",
    "Combined RQ4 warning outcomes for histories significant "
    "in both RQ3 performance metrics.",
)

export_both(
    publication_table,
    "rq3_common_histories_rq4_warning_summary",
    "Actionable, too-early, too-late, missed, and warning outcomes "
    "for common RQ3-significant histories.",
)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------
print("Common RQ3 significant histories:", len(common_histories))
print("RQ4 outcome rows:", len(selected_long))
print("Denominator per configuration:",
      sorted(selected_long["total_events"].unique()))
print("Output folder:", RQ4)

print("\nHistories significant in both Accuracy and Macro-F1:")
display(common_histories)

print("\nDetailed RQ4 outcomes:")
display(selected_long)

print("\nCompact warning summary:")
display(publication_table)